# 🔬 Notebook 4: Triangulación de Insights
## Análisis de Churn en Mercados Emergentes

---

### Objetivo
Sintetizar los hallazgos cuantitativos del análisis de datos con los insights cualitativos de las entrevistas, generando recomendaciones accionables para el diseño de productos fintech.

### Contenido
1. Síntesis de hallazgos cuantitativos
2. Revisión de insights cualitativos
3. Triangulación: De los datos a la historia
4. Modelo de desconfianza acumulativa
5. Implicaciones para diseño de producto
6. Recomendaciones de intervención
7. Limitaciones y próximos pasos

## 1. Configuración Inicial

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from datetime import datetime

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Rutas
DATA_PROCESSED = Path('../data/processed')
QUALITATIVE = Path('../qualitative')
OUTPUTS = Path('../outputs/figures')

print("✅ Configuración completada")

## 2. Síntesis de Hallazgos Cuantitativos

In [ ]:
# Cargar datos
df = pd.read_csv(DATA_PROCESSED / 'churn_features.csv')

print("📊 SÍNTESIS DE HALLAZGOS CUANTITATIVOS")
print("="*70)

# 1. Tasa de churn general
churn_rate = (df['Churn'] == 'Yes').mean() * 100
print(f"\n1. TASA DE CHURN GENERAL")
print(f"   • Total de clientes: {len(df):,}")
print(f"   • Tasa de churn: {churn_rate:.1f}%")

# 2. Churn por contrato
contract_churn = df.groupby('Contract').apply(
    lambda x: (x['Churn'] == 'Yes').mean() * 100
).sort_values(ascending=False)

print(f"\n2. CHURN POR TIPO DE CONTRATO")
for contract, rate in contract_churn.items():
    print(f"   • {contract}: {rate:.1f}%")

print(f"   → Riesgo relativo: {contract_churn['Month-to-month']/contract_churn['Two year']:.1f}x")

# 3. Churn por tenure
print(f"\n3. CHURN POR GRUPO DE TENURE")
tenure_churn = df.groupby('tenure_group').apply(
    lambda x: (x['Churn'] == 'Yes').mean() * 100
)
for group, rate in tenure_churn.items():
    print(f"   • {group}: {rate:.1f}%")

# 4. Cargos
print(f"\n4. CARGOS PROMEDIO")
churner_charges = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()
non_churner_charges = df[df['Churn'] == 'No']['MonthlyCharges'].mean()
print(f"   • Churners: ${churner_charges:.2f}")
print(f"   • No-churners: ${non_churner_charges:.2f}")
print(f"   → Diferencia: +{(churner_charges/non_churner_charges - 1)*100:.1f}%")

In [ ]:
# Crear visualización resumen
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Distribución de churn
ax1 = axes[0, 0]
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax1.pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%', 
        colors=colors, explode=(0, 0.05), startangle=90)
ax1.set_title('Distribución de Churn', fontsize=12, fontweight='bold')

# 2. Churn por contrato
ax2 = axes[0, 1]
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax2.bar(contract_churn.index, contract_churn.values, color=colors)
ax2.set_title('Churn por Tipo de Contrato', fontsize=12, fontweight='bold')
ax2.set_ylabel('Tasa de Churn (%)')
ax2.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, contract_churn.values):
    ax2.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Churn por tenure
ax3 = axes[1, 0]
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
bars = ax3.bar(tenure_churn.index, tenure_churn.values, color=colors)
ax3.set_title('Churn por Grupo de Tenure', fontsize=12, fontweight='bold')
ax3.set_ylabel('Tasa de Churn (%)')
for bar, val in zip(bars, tenure_churn.values):
    ax3.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. Cargos
ax4 = axes[1, 1]
categories = ['Churners', 'No Churners']
values = [churner_charges, non_churner_charges]
colors = ['#e74c3c', '#2ecc71']
bars = ax4.bar(categories, values, color=colors)
ax4.set_title('Cargos Mensuales Promedio', fontsize=12, fontweight='bold')
ax4.set_ylabel('Cargo Mensual ($)')
for bar, val in zip(bars, values):
    ax4.annotate(f'${val:.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Resumen de Hallazgos Cuantitativos', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS / '23_quantitative_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Revisión de Insights Cualitativos

In [ ]:
print("\n📝 SÍNTESIS DE INSIGHTS CUALITATIVOS")
print("="*70)

print("\n📊 DATOS DE ENTREVISTAS:")
print("   • Participantes: 5 comerciantes venezolanos")
print("   • Duración: 30-45 minutos cada entrevista")
print("   • Enfoque: Experiencias de abandono de herramientas digitales")

print("\n📋 PATRONES IDENTIFICADOS:")

patterns = {
    'Opacidad en procesos': {'frecuencia': '5/5', 'severidad': 'Alta'},
    'Comunicación no proactiva': {'frecuencia': '4/5', 'severidad': 'Alta'},
    'Cambios sin aviso': {'frecuencia': '3/5', 'severidad': 'Media-Alta'},
    'Soporte insuficiente': {'frecuencia': '4/5', 'severidad': 'Alta'},
    'Discrepancia promesa-realidad': {'frecuencia': '4/5', 'severidad': 'Alta'}
}

for pattern, data in patterns.items():
    print(f"\n   • {pattern}")
    print(f"     Frecuencia: {data['frecuencia']} | Severidad: {data['severidad']}")

print("\n📅 TIMELINE DE ABANDONO:")
timeline_data = [
    ('María', '8 meses', 'Incidente no resuelto'),
    ('Carlos', '14 meses', 'Falla sin comunicación'),
    ('Ana', '5 meses', 'Reglas ocultas'),
    ('José', '6 meses', 'Cambio de interfaz'),
    ('Luisa', '4 meses', 'Problema de canje')
]

for name, time, trigger in timeline_data:
    print(f"   • {name}: {time} - {trigger}")

print(f"\n   → 60% abandonó en los primeros 6 meses")

In [ ]:
# Visualización de timeline cualitativo
fig, ax = plt.subplots(figsize=(14, 6))

names = ['María', 'Carlos', 'Ana', 'José', 'Luisa']
times = [8, 14, 5, 6, 4]
triggers = ['Incidente no resuelto', 'Falla sin comunicación', 
            'Reglas ocultas', 'Cambio de interfaz', 'Problema de canje']

colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c']

bars = ax.barh(names, times, color=colors, edgecolor='white', linewidth=2)

for bar, time, trigger in zip(bars, times, triggers):
    ax.annotate(f'{time} meses\n{trigger}',
                xy=(time + 0.5, bar.get_y() + bar.get_height()/2),
                ha='left', va='center', fontsize=10)

# Zona crítica
ax.axvline(6, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.axvspan(0, 6, alpha=0.1, color='red')
ax.annotate('Zona Crítica\n(60% de abandonos)', xy=(3, 4.5), 
            fontsize=11, ha='center', color='red', fontweight='bold')

ax.set_xlabel('Tiempo hasta Abandono (meses)', fontsize=12)
ax.set_title('Timeline de Abandono - Entrevistas Cualitativas', fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '24_qualitative_timeline.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Triangulación: De los Datos a la Historia

In [ ]:
print("\n" + "="*70)
print("🔬 TRIANGULACIÓN: DE LOS DATOS A LA HISTORIA")
print("="*70)

print("\n📊 DIMENSIÓN 1: VENTANA CRÍTICA")
print("-"*50)
print("\n   CUANTITATIVO:")
print(f"   • 47.4% de clientes en 0-12 meses abandona")
print(f"   • Hazard rate máximo en primeros meses")
print(f"   • Mediana de supervivencia contratos mensuales: ~10 meses")

print("\n   CUALITATIVO:")
print(f"   • 60% de entrevistados abandonó en 0-6 meses")
print(f"   • Todos describen 'acumulación de incidentes'")
print(f"   • Abandono es decisión tipping point, no gradual")

print("\n   TRIANGULACIÓN:")
print(f"   El modelo identifica CUÁNDO (meses 6-10)")
print(f"   Las entrevistas explican POR QUÉ (acumulación de incidentes sin explicación)")

In [ ]:
print("\n📊 DIMENSIÓN 2: TIPO DE CONTRATO Y COMPROMISO")
print("-"*50)
print("\n   CUANTITATIVO:")
print(f"   • Month-to-month: 42.7% churn")
print(f"   • One year: 11.3% churn")
print(f"   • Two year: 2.8% churn")
print(f"   • Riesgo relativo: 3.2x")

print("\n   CUALITATIVO:")
print(f"   • 'Yo pensaba que éramos socios' - Ana")
print(f"   • 'Me sentí engañada' - María")
print(f"   • 'Trátame como adulto, no como tonto' - Carlos")

print("\n   TRIANGULACIÓN:")
print(f"   El contrato formal reduce churn, pero el 'contrato emocional' importa más.")
print(f"   Cuando la empresa viola la confianza, el contrato formal se vuelve inverso:")
print(f"   el usuario se siente atrapado y resentido.")

In [ ]:
print("\n📊 DIMENSIÓN 3: CARGOS Y VALOR PERCIBIDO")
print("-"*50)
print("\n   CUANTITATIVO:")
print(f"   • Churners pagan ${churner_charges:.2f} promedio")
print(f"   • No-churners pagan ${non_churner_charges:.2f} promedio")
print(f"   • Diferencia: +{(churner_charges/non_churner_charges - 1)*100:.1f}%")

print("\n   CUALITATIVO:")
print(f"   • Solo 1/5 mencionó precio como factor primario")
print(f"   • 5/5 mencionaron transparencia/valor como factor")
print(f"   • '¿Qué mejoras?' - Carlos sobre incremento de precios")

print("\n   TRIANGULACIÓN:")
print(f"   El churn no es función de PRECIO, es función de VALOR PERCIBIDO.")
print(f"   La opacidad reduce el valor percibido independientemente del precio.")

In [ ]:
# Crear tabla de triangulación
triangulation_data = {
    'Dimensión': ['Ventana Crítica', 'Contrato', 'Cargos', 'Método Pago'],
    'Hallazgo Cuantitativo': [
        '47% abandona en 0-12m',
        'Mensual = 3x más churn',
        'Churners pagan +21.5%',
        'Auto-pago = menos churn'
    ],
    'Hallazgo Cualitativo': [
        '60% abandona en 0-6m\nAcumulación de incidentes',
        'Contrato emocional\nimporta más que formal',
        'Problema de valor,\nno de precio',
        '"Lo conocido" es\nmás confiable'
    ],
    'Implicación': [
        'Intervenir en mes 3-6',
        'Compromiso bilateral',
        'Transparencia > Precio',
        'Integrar con lo familiar'
    ]
}

triangulation_df = pd.DataFrame(triangulation_data)

print("\n📊 TABLA DE TRIANGULACIÓN:")
print("="*80)
display(triangulation_df)

## 5. Modelo de Desconfianza Acumulativa

In [ ]:
print("\n" + "="*70)
print("📈 MODELO DE DESCONFIANZA ACUMULATIVA")
print("="*70)

model_text = """
┌─────────────────────────────────────────────────────────────────┐
│  [DESCONFIANZA INSTITUCIONAL PREEXISTENTE]                      │
│              ↓                                                  │
│  [Usuario adopta herramienta con esperanza pero escepticismo]   │
│              ↓                                                  │
│  [Incidente 1 + Opacidad] → [Reducción de "pool de buena        │
│              ↓                       voluntad"]                 │
│  [Incidente 2 + Opacidad] → [Más reducción]                     │
│              ↓                                                  │
│  [Incidente N (trigger)] → ["Ya no vale la pena el estrés"]     │
│              ↓                                                  │
│  [ABANDONO]                                                     │
└─────────────────────────────────────────────────────────────────┘
"""

print(model_text)

print("\n📋 CARACTERÍSTICAS DEL MODELO:")
print("   1. No es lineal: El abandono es una decisión 'tipping point'")
print("   2. Es emocional: La clave es sentirse 'engañado'")
print("   3. Es reversible hasta el trigger: La intervención funciona antes")
print("   4. Es cultural: En mercados de desconfianza, el pool inicial es menor")

In [ ]:
# Visualización del modelo de desconfianza
fig, ax = plt.subplots(figsize=(14, 8))

# Representación visual del pool de buena voluntad
months = np.arange(0, 15)
base_goodwill = 100  # Pool inicial en mercados de alta confianza
emergent_goodwill = 70  # Pool inicial en mercados emergentes

# Decay rates (diferentes escenarios)
decay_transparent = 0.02  # Con transparencia
decay_opaque = 0.08  # Con opacidad

goodwill_transparent = emergent_goodwill * np.exp(-decay_transparent * months)
goodwill_opaque = emergent_goodwill * np.exp(-decay_opaque * months)

ax.fill_between(months, goodwill_transparent, alpha=0.3, color='#2ecc71', label='Con transparencia')
ax.plot(months, goodwill_transparent, color='#2ecc71', linewidth=3)

ax.fill_between(months, goodwill_opaque, alpha=0.3, color='#e74c3c', label='Con opacidad')
ax.plot(months, goodwill_opaque, color='#e74c3c', linewidth=3)

# Línea de abandono
ax.axhline(30, color='orange', linestyle='--', linewidth=2, label='Umbral de abandono')

# Marcar zona de intervención
ax.axvspan(3, 6, alpha=0.2, color='blue', label='Ventana de intervención')

# Anotaciones
ax.annotate('Pool de "Buena Voluntad"\n(Mercados Emergentes)', xy=(0, 75), fontsize=11)
ax.annotate('Transparencia\npreserva confianza', xy=(10, 65), fontsize=10, color='#2ecc71')
ax.annotate('Opacidad\nerosiona confianza', xy=(8, 30), fontsize=10, color='#e74c3c')
ax.annotate('Abandono', xy=(10, 27), fontsize=10, color='orange', fontweight='bold')

ax.set_xlabel('Meses de uso', fontsize=12)
ax.set_ylabel('Pool de Buena Voluntad (%)', fontsize=12)
ax.set_title('Modelo de Desconfianza Acumulativa', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right')
ax.set_xlim(0, 14)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / '25_distrust_model.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Implicaciones para Diseño de Producto

In [ ]:
print("\n" + "="*70)
print("💡 IMPLICACIONES PARA DISEÑO DE PRODUCTO")
print("="*70)

print("\n📊 PRINCIPIOS DE DISEÑO DE CONFIANZA:")
print("-"*50)

principles = [
    ("Visibilidad", "Mostrar siempre el estado de procesos (en cola, procesando, completado)"),
    ("Explicabilidad", "Cada cargo, cada cambio, cada decisión debe tener explicación accesible"),
    ("Soporte humano", "Para problemas, siempre ofrecer opción de hablar con persona real"),
    ("Retroceso fácil", "Permitir 'deshacer' acciones por un período"),
    ("History claro", "El usuario debe poder ver su historial completo")
]

for principle, description in principles:
    print(f"\n   • {principle}")
    print(f"     {description}")

In [ ]:
print("\n📊 INTERVENCIÓN PROACTIVA (Meses 3-6):")
print("-"*50)

interventions = [
    ("Mes 1", "Onboarding con expectativas claras", "Reducir sorpresas"),
    ("Mes 2", "Check-in proactivo ('¿cómo te va?')", "Detectar fricciones temprano"),
    ("Mes 3", "Tutorial de features no usadas", "Aumentar valor percibido"),
    ("Mes 6", "Oferta de compromiso (descuento por anualidad)", "Aumentar switching cost")
]

print("\n   Momento  | Acción                               | Objetivo")
print("   " + "-"*65)
for moment, action, objective in interventions:
    print(f"   {moment:8} | {action:36} | {objective}")

In [ ]:
print("\n📊 COMUNICACIÓN TRANSPARENTE:")
print("-"*50)

comms = [
    ("Cambio de términos", "Aviso 30 días + explicación simple", "'Consulte términos actualizados'"),
    ("Incremento de precio", "Justificación clara + opción de downgrade", "Esconder en fine print"),
    ("Falla del sistema", "Comunicación inmediata + ETA de resolución", "Silencio hasta que pregunten"),
    ("Error del usuario", "Explicación de qué pasó + cómo evitarlo", "Mensajes genéricos de error")
]

print("\n   Situación            | Acción                              | Evitar")
print("   " + "-"*75)
for situation, action, avoid in comms:
    print(f"   {situation:20} | {action:36} | {avoid}")

## 7. Recomendaciones Finales

In [ ]:
print("\n" + "="*70)
print("🎯 RECOMENDACIONES FINALES")
print("="*70)

recommendations = [
    {
        'titulo': '1. Intervención Proactiva',
        'prioridad': 'ALTA',
        'impacto': 'Reducción estimada 15-25% churn temprano',
        'accion': 'Implementar check-ins automatizados en meses 2, 4, 6'
    },
    {
        'titulo': '2. Transparencia Radical',
        'prioridad': 'ALTA',
        'impacto': 'Aumento de confianza y LTV',
        'accion': 'Dashboard de estado en tiempo real + historial completo'
    },
    {
        'titulo': '3. Comunicación Anticipada',
        'prioridad': 'MEDIA',
        'impacto': 'Reducción de sorpresas negativas',
        'accion': 'Sistema de notificaciones proactivas para cambios'
    },
    {
        'titulo': '4. Soporte Humano Accesible',
        'prioridad': 'ALTA',
        'impacto': 'Resolución de problemas antes del tipping point',
        'accion': 'Canal directo a humano para problemas críticos'
    },
    {
        'titulo': '5. Incentivos de Compromiso',
        'prioridad': 'MEDIA',
        'impacto': 'Reducción de riesgo en contratos mensuales',
        'accion': 'Beneficios claros por migrar a contratos anuales'
    }
]

for rec in recommendations:
    print(f"\n{rec['titulo']}")
    print(f"   Prioridad: {rec['prioridad']}")
    print(f"   Impacto: {rec['impacto']}")
    print(f"   Acción: {rec['accion']}")

## 8. Limitaciones del Estudio

In [ ]:
print("\n" + "="*70)
print("⚠️ LIMITACIONES DEL ESTUDIO")
print("="*70)

limitations = [
    ("Dataset sintético", "Telco Churn es un dataset académico, no datos reales de fintech. Los patrones pueden diferir en contexto financiero."),
    ("Sample cualitativo pequeño", "n=5 entrevistas no es representativo estadísticamente. Los hallazgos son exploratorios, no confirmatorios."),
    ("Contexto específico", "Venezuela tiene características únicas de desconfianza institucional. Los hallazgos pueden no generalizar a otros mercados."),
    ("Sesgo de memoria", "Las entrevistas dependen de memoria de eventos pasados. Los participantes pueden recordar sesgadamente."),
    ("Ausencia de validación", "Los insights no han sido validados con datos de comportamiento real en Cashea.")
]

for limitation, description in limitations:
    print(f"\n   • {limitation}")
    print(f"     {description}")

In [ ]:
print("\n" + "="*70)
print("📈 PRÓXIMOS PASOS RECOMENDADOS")
print("="*70)

next_steps = [
    "1. Validación cuantitativa: Recopilar datos de churn de Cashea para validar patrones",
    "2. Estudio longitudinal: Seguir usuarios desde adopción por 12 meses",
    "3. A/B testing: Probar estrategias de comunicación proactiva",
    "4. Encuesta de confianza: Desarrollar instrumento para medir 'pool de buena voluntad'",
    "5. Benchmarking: Comparar con métricas de industria fintech LATAM"
]

for step in next_steps:
    print(f"\n   {step}")

## 9. Conclusión Final

In [ ]:
print("\n" + "="*70)
print("🏁 CONCLUSIÓN FINAL")
print("="*70)

conclusion = """
El análisis de churn en mercados emergentes requiere ir más allá de las métricas 
tradicionales. La desconfianza institucional crea un contexto donde la transparencia 
no es un "nice-to-have" sino un factor de supervivencia.

EL MODELO IDENTIFICA CUÁNDO ABANDONAN LOS USUARIOS.
LAS ENTREVISTAS EXPLICAN POR QUÉ.

JUNTOS, NOS DICEN CÓMO INTERVENIR:
• Proactivamente
• Con transparencia
• Construyendo confianza antes de que se erosione

Para Cashea y otras fintech en mercados emergentes, la lección central es:
El churn no se previene con descuentos, se previene con confianza.
"""

print(conclusion)

print("\n" + "="*70)
print(f"📅 Análisis completado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("📁 Archivos generados en: data/processed/, outputs/figures/""")
print("="*70)